# Apache Spark Architecture
**Day 2 — Technology Module**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>Core Insight:</strong> Spark is the industry standard for distributed data processing.
Senior data engineers know the execution model — not just "it's fast" but WHY it's fast
and what makes it slow. Key: lazy evaluation, DAG optimization, shuffle is the enemy.
</div>

### The Execution Model
```
Driver Program (your Python/Scala code)
    ↓  creates
SparkSession / SparkContext
    ↓  builds
DAG of Stages (Directed Acyclic Graph)
    ↓  each stage is a set of
Tasks (one per partition, runs in parallel)
    ↓  executed on
Executors (JVMs on worker nodes)
```

### RDD → DataFrame → Dataset
- **RDD:** Low-level, untyped. Use only for legacy code.
- **DataFrame:** Optimized by Catalyst query optimizer. Use this always.
- **Dataset:** Typed DataFrame (Scala/Java only). Python uses DataFrame.

## Transformations vs Actions

**Transformations** build the DAG but execute nothing:
```python
filtered = df.filter(df.cpu > 80)          # lazy
grouped  = filtered.groupBy("server_id")    # lazy
averaged = grouped.agg({"cpu": "avg"})      # lazy
# Nothing has run yet — DAG is just a plan
```

**Actions** trigger execution:
```python
averaged.show()                              # NOW the DAG runs
averaged.write.parquet("s3://output/")       # another action
count = averaged.count()                     # another action
```

The lazy model lets Spark optimize the *entire plan* before running anything — filter pushdown, join reordering, etc.

In [ ]:
# LOCAL SIMULATION — pandas equivalent of Spark transformations + actions
# In production, replace spark.read/write and spark.sql with the PySpark equivalents.

import pandas as pd
import numpy as np

# Simulate a large telemetry dataset (what would come from S3 in Spark)
np.random.seed(42)
n = 50_000
raw_telemetry = pd.DataFrame({
    'server_id':       [f'srv-{i:03d}' for i in np.random.randint(1, 101, n)],
    'cpu_utilization': np.random.uniform(20, 100, n).round(1),
    'mem_utilization': np.random.uniform(30, 95, n).round(1),
    'region':          np.random.choice(['us-east', 'us-west', 'eu-west'], n),
    'year': '2026', 'month': '02',
})

# ── "Transformations" (lazy in Spark, eager in pandas) ───────────────────────
# Step 1: filter
filtered = raw_telemetry[raw_telemetry['cpu_utilization'] > 80]

# Step 2: add derived column (withColumn in Spark)
filtered = filtered.assign(
    alert_tier=pd.cut(filtered['cpu_utilization'],
                      bins=[80, 90, 95, 100],
                      labels=['warning', 'high', 'critical'])
)

# Step 3: groupBy + agg
result = (filtered.groupby(['region', 'alert_tier'])
    .agg(
        server_count=('server_id', pd.Series.nunique),
        avg_cpu=('cpu_utilization', 'mean'),
        max_cpu=('cpu_utilization', 'max'),
    ).round(2).reset_index())

# ── "Action" — materialize and display ───────────────────────────────────────
print(f"Input rows: {len(raw_telemetry):,}")
print(f"After filter (cpu > 80): {len(filtered):,} rows")
print()
print("Aggregated result (Spark equivalent: .show()):")
print(result.to_string(index=False))
print()
print("Spark version of this pipeline:")
print("""
  df = spark.read.parquet("s3://telemetry/")
  filtered = df.filter(df.cpu_utilization > 80)
  result = filtered.groupBy("region", "alert_tier").agg(
      F.countDistinct("server_id").alias("server_count"),
      F.avg("cpu_utilization").alias("avg_cpu"),
  )
  result.show()   # ← this triggers execution of the entire DAG
""")

## Partitions = Parallelism

```
Dataset
  ├── Partition 1  →  Task 1  →  Executor A
  ├── Partition 2  →  Task 2  →  Executor B
  ├── Partition 3  →  Task 3  →  Executor C
  └── Partition 4  →  Task 4  →  Executor A
```

More partitions = more parallelism. But too many = scheduling overhead.

**Rule of thumb:** 2-4 partitions per CPU core. 128-256 MB per partition.

```python
# Check and adjust partitions
df.rdd.getNumPartitions()       # see current count

df.repartition(100)             # full shuffle — use when increasing or balancing
df.coalesce(10)                 # local merge — use when decreasing (no shuffle)

# Write partitioned by column (downstream query pruning)
df.write.partitionBy("year", "month").parquet("s3://output/")
```

In [ ]:
# Partition simulation — illustrate repartition vs coalesce behavior

import pandas as pd

# Simulate 12 "partitions" of data (in Spark, each would be a separate task)
data_by_partition = []
for part_id in range(12):
    size = 1000 + (part_id % 3) * 500   # simulate skew: some partitions are bigger
    data_by_partition.append({
        'partition': part_id,
        'row_count': size,
        'size_mb': round(size * 0.0001, 3)  # 100 bytes per row approx
    })

df_parts = pd.DataFrame(data_by_partition)
print("Current partitions (simulated):")
print(df_parts.to_string(index=False))
print(f"Total rows: {df_parts['row_count'].sum():,}")
print(f"Avg partition size: {df_parts['row_count'].mean():.0f} rows")
print(f"Max skew ratio: {df_parts['row_count'].max()/df_parts['row_count'].min():.1f}x")
print()
print("repartition(6):  full shuffle, 6 balanced partitions — expensive but balanced")
print("coalesce(6):     merge locally, 6 partitions no shuffle — cheap but may be uneven")
print()
print("When to use each:")
print("  repartition: increasing partition count OR fixing data skew (groupBy skew)")
print("  coalesce:    reducing partition count before write (avoid many tiny files)")

## What Causes a Shuffle?

A shuffle = moving data across the network between executors. It's the #1 Spark performance bottleneck.

**Operations that cause shuffle:**
- `groupBy()` — data for the same key must be on the same executor
- `join()` — join keys must be co-located
- `repartition()` — explicit redistribution
- `distinct()` — de-duplication requires comparing across all partitions

**How to minimize shuffle:**
1. **Filter early** — reduce data before shuffle operations
2. **Broadcast join** — replicate a small table to all executors (avoids shuffle entirely)
3. **Partition wisely** — write data partitioned by the join key you'll use most
4. **Cache before re-use** — if you'll run multiple actions on the same DataFrame, `df.cache()` avoids recomputing from scratch

```python
# Broadcast join — small lookup table (< few hundred MB)
from pyspark.sql import functions as F
server_metadata = spark.read.parquet("s3://metadata/servers/")   # small
df_enriched = df.join(F.broadcast(server_metadata), on="server_id", how="left")
```

## Interview Q&A

**Q: What is the Catalyst optimizer?**
A: Spark's query optimizer — transforms the logical plan (what you wrote) into an optimized physical plan (what Spark executes). It pushes filters down, reorders joins, eliminates unnecessary columns. Same role as a SQL query planner.

**Q: What's the difference between `repartition` and `coalesce`?**
A: `repartition` does a full shuffle — good for increasing partitions or fixing data skew. `coalesce` merges existing partitions locally — good for reducing partition count cheaply before a write. coalesce can't increase partition count.

**Q: Explain lazy evaluation.**
A: Transformations build a DAG but don't run. Only actions trigger execution. This lets Spark optimize the entire plan before running — filter pushdown, join reordering — things you can't do if you execute eagerly.

**Q: How many partitions should you have?**
A: 2-4 per CPU core. 128-256 MB per partition is typical. Too few = underutilized cores. Too many = excessive scheduling overhead. Adjust with `repartition()` or `coalesce()`.

**Q: What is data skew and how do you fix it?**
A: When one partition has far more data than others (one key dominates). Fix: (1) filter or pre-aggregate the skewed key separately, (2) salting — add a random suffix to the key to distribute it, (3) use a broadcast join if the table is small enough.

## The Citi Angle

**Spark was the natural evolution of our telemetry pipeline.** As data volume grew from gigabytes to terabytes, pandas jobs hit memory limits. Spark's partition model matched our data naturally — partition by server group, then process each group in parallel.

**The capacity forecasting pipeline:**
```
1. Spark reads 6 months of daily Parquet from S3 (partitioned by date)
   → Partition pruning: only reads relevant months
2. groupBy("server_id").agg(rolling averages, percentiles)
   → One shuffle — unavoidable
3. join with server_metadata (broadcast — metadata is < 50MB)
   → No shuffle — broadcast avoids it
4. Prophet forecasting model applied per server
   → mapPartitions: run Python model on each partition locally
5. Write predictions back to S3 (partitioned by forecast_date)
   → coalesce(50) before write — prevent small files
```

**Interview line:** *"Shuffle is Spark's most expensive operation. In the Citi forecasting pipeline, I reduced shuffle by broadcasting server metadata (avoiding a 6B-row join shuffle), partitioning input by date (partition pruning), and coalescing before write. Pipeline time dropped from 45 minutes to 12 minutes."*